In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# QUESTION 1
Implement a Feedforward Neural Network (FNN) From Scratch

Architecture:

Input Layer → (n features)

Hidden Layer → 2 neurons (ReLU)

Output Layer → 1 neuron (Sigmoid)

Loss → Binary Cross Entropy

Optimizer → Batch Gradient Descent

In [2]:
df = pd.read_csv("titanic_selected_features.csv")

X = df.drop("Survived", axis=1).values
y = df["Survived"].values.reshape(-1,1)

In [3]:
mean = X.mean(axis=0)
std = X.std(axis=0)

X = (X - mean) / std

In [4]:
np.random.seed(42)

indices = np.random.permutation(len(X))

train_end = int(0.7 * len(X))
val_end = int(0.85 * len(X))

train_idx = indices[:train_end]
val_idx = indices[train_end:val_end]
test_idx = indices[val_end:]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

### PART (a)

In [5]:
def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(float)

def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

In [6]:
def initialize(n_input, n_hidden):
    np.random.seed(42)
    W1 = np.random.randn(n_input, n_hidden) * 0.01
    b1 = np.zeros((1, n_hidden))
    W2 = np.random.randn(n_hidden, 1) * 0.01
    b2 = np.zeros((1, 1))
    
    return W1, b1, W2, b2

In [7]:
def forward(X, W1, b1, W2, b2):
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2)
    
    return Z1, A1, Z2, A2

### PART (b)

In [8]:
def compute_loss(y, y_hat):
    m = y.shape[0]
    loss = -(1/m) * np.sum(
        y*np.log(y_hat + 1e-8) +
        (1-y)*np.log(1 - y_hat + 1e-8)
    )
    return loss

### PART (c)

In [9]:
def backward(X, y, Z1, A1, A2, W2):
    m = y.shape[0]
    
    dZ2 = A2 - y
    dW2 = (1/m) * np.dot(A1.T, dZ2)
    db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)
    
    dZ1 = np.dot(dZ2, W2.T) * relu_derivative(Z1)
    dW1 = (1/m) * np.dot(X.T, dZ1)
    db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)
    
    return dW1, db1, dW2, db2

In [10]:
def train(X, y, n_hidden=2, lr=0.01, epochs=100):
    n_input = X.shape[1]
    W1, b1, W2, b2 = initialize(n_input, n_hidden)
    
    losses = []
    
    for i in range(epochs):
        Z1, A1, Z2, A2 = forward(X, W1, b1, W2, b2)
        loss = compute_loss(y, A2)
        losses.append(loss)
        
        dW1, db1, dW2, db2 = backward(X, y, Z1, A1, A2, W2)
        
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2
        
    return W1, b1, W2, b2, losses

# QUESTION 2
Hyperparameter Tuning

Parameters:

Epochs → 10, 15, 25, 100

Hidden units → 4, 8, 16

Learning rate → 0.2, 0.1, 0.01

In [11]:
epochs_list = [10, 15, 25, 100]
hidden_list = [4, 8, 16]
lr_list = [0.2, 0.1, 0.01]

best_val_loss = float('inf')
best_params = None

for e in epochs_list:
    for h in hidden_list:
        for lr in lr_list:
            W1, b1, W2, b2, _ = train(X_train, y_train, h, lr, e)
            _, _, _, val_pred = forward(X_val, W1, b1, W2, b2)
            val_loss = compute_loss(y_val, val_pred)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_params = (e, h, lr)

print("Best Hyperparameters:", best_params)

Best Hyperparameters: (100, 16, 0.2)


In [12]:
e, h, lr = best_params

W1, b1, W2, b2, _ = train(X_train, y_train, h, lr, e)

_, _, _, test_pred = forward(X_test, W1, b1, W2, b2)

test_loss = compute_loss(y_test, test_pred)

test_predictions = (test_pred > 0.5).astype(int)
accuracy = np.mean(test_predictions == y_test)

print("Test Loss:", test_loss)
print("Test Accuracy:", accuracy)

Test Loss: 0.49404384324542083
Test Accuracy: 0.835820895522388


# QUESTION 3

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim

In [14]:
class FNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(FNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x

In [17]:
model = FNN(X_train.shape[1], h)

criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=lr)

X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)

for epoch in range(e):
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

In [18]:
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.FloatTensor(y_test)

with torch.no_grad():
    outputs = model(X_test_t)
    loss = criterion(outputs, y_test_t)
    predictions = (outputs > 0.5).float()
    accuracy = (predictions == y_test_t).float().mean()

print("PyTorch Test Loss:", loss.item())
print("PyTorch Test Accuracy:", accuracy.item())

PyTorch Test Loss: 0.43459275364875793
PyTorch Test Accuracy: 0.8283582329750061


# Question 4

In [36]:
class GeneralizedANN:
    def __init__(self, layer_sizes, learning_rate):
        
        self.lr = learning_rate
        self.L = len(layer_sizes) - 1
        
        self.W = []
        self.b = []
        
        for i in range(self.L):
            self.W.append(np.random.randn(layer_sizes[i], layer_sizes[i+1]) * 0.01)
            self.b.append(np.zeros((1, layer_sizes[i+1])))

        def forward(self, X):
            self.A = [X]
            self.Z = []
            
            for l in range(self.L):
                Z = self.A[l] @ self.W[l] + self.b[l]
                A = sigmoid(Z)
                
                self.Z.append(Z)
                self.A.append(A)
            
            return self.A[-1]

        def backward(self, X, y):
            m = X.shape[0]
            
            dZ = self.A[-1] - y
            dW = [None] * self.L
            db = [None] * self.L
            
            for l in reversed(range(self.L)):
                dW[l] = (self.A[l].T @ dZ) / m
                db[l] = np.sum(dZ, axis=0, keepdims=True) / m
                
                if l > 0:
                    dA = dZ @ self.W[l].T
                    dZ = dA * sigmoid_derivative(self.A[l])
            
            for l in range(self.L):
                self.W[l] -= self.lr * dW[l]
                self.b[l] -= self.lr * db[l]

        def train(self, X, y, epochs):
            losses = []
            
            for epoch in range(epochs):
                y_pred = self.forward(X)
                loss = cross_entropy_loss(y, y_pred)
                self.backward(X, y)
                losses.append(loss)
                
                print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")
            
            return losses

